In [1]:
!pip install nltk torch pandas scikit-learn -q --break-system-packages 2>/dev/null || pip install nltk torch pandas scikit-learn -q

In [2]:
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.stem.porter import PorterStemmer

nltk.download('punkt')
nltk.download('punkt_tab')

stemmer = PorterStemmer()

def tokenize(sentence):
    return nltk.word_tokenize(sentence)

def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, all_words):
    tokenized_sentence = [stem(w) for w in tokenized_sentence]
    bag = np.zeros(len(all_words), dtype=np.float32)
    for idx, w in enumerate(all_words):
        if w in tokenized_sentence:
            bag[idx] = 1.0
    return bag

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
df = pd.read_csv("dataset_intent_jadwal_kuliah.csv")
df = df.fillna("")
print(df.shape)
df.head(10)

(152, 3)


,intent,text,entity
0,jadwal_hari_ini,Apa saja jadwal kuliah hari ini?,
1,jadwal_hari_ini,Hari ini kuliah apa saja?,
2,jadwal_hari_ini,Kelas apa hari ini?,
3,jadwal_hari_ini,Ada mata kuliah apa hari ini?,
4,jadwal_hari_ini,Siapa dosennya hari ini dan kuliah jam berapa?,
5,jadwal_hari_tertentu,Jadwal kuliah hari Senin apa saja?,hari=Senin
6,jadwal_hari_tertentu,Kuliah di hari Senin ada apa aja?,hari=Senin
7,jadwal_hari_tertentu,Hari Senin ada kelas apa?,hari=Senin
8,jadwal_hari_tertentu,Apa jadwal kuliah di hari Senin?,hari=Senin
9,jadwal_hari_tertentu,Jadwal hari Senin itu apa saja?,hari=Senin


In [4]:
all_words = []
tags = []
xy = []

ignore_words = ['?', '.', '!', ',']

for _, row in df.iterrows():
    tag = row['intent']
    text = row['text']
    if tag not in tags:
        tags.append(tag)
    w = tokenize(text)
    all_words.extend(w)
    xy.append((w, tag))

all_words = [stem(w) for w in all_words if w not in ignore_words]
all_words = sorted(set(all_words))
tags = sorted(set(tags))

print(len(xy), "patterns")
print(len(tags), "tags:", tags)
print(len(all_words), "unique stemmed words")

152 patterns
6 tags: ['jadwal_hari_ini', 'jadwal_hari_tertentu', 'mata_kuliah_berdasarkan_lokasi', 'mata_kuliah_dosen', 'mata_kuliah_per_semester', 'mata_kuliah_semester_tipe']
87 unique stemmed words


In [5]:
X_train = []
y_train = []

for (pattern_sentence, tag) in xy:
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    label = tags.index(tag)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(X_train.shape, y_train.shape)

(152, 87) (152,)


In [6]:
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = X_train
        self.y_data = y_train

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

In [7]:
num_epochs = 1000
batch_size = 8
learning_rate = 0.001
input_size = len(all_words)
hidden_size = 8
output_size = len(tags)

print("input_size:", input_size)
print("output_size:", output_size)

dataset = ChatDataset()
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

model = NeuralNet(input_size, hidden_size, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

input_size: 87
output_size: 6
device: cpu


In [8]:
for epoch in range(num_epochs):
    for (words, labels) in train_loader:
        words = words.to(device).float()
        labels = labels.to(device).long()

        outputs = model(words)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print(f'Final loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 0.0028


Epoch [200/1000], Loss: 0.0008


Epoch [300/1000], Loss: 0.0001


Epoch [400/1000], Loss: 0.0001


Epoch [500/1000], Loss: 0.0000


Epoch [600/1000], Loss: 0.0000


Epoch [700/1000], Loss: 0.0000


Epoch [800/1000], Loss: 0.0000


Epoch [900/1000], Loss: 0.0000


Epoch [1000/1000], Loss: 0.0000
Final loss: 0.0000


In [9]:
model.eval()
correct = 0
with torch.no_grad():
    for i in range(len(X_train)):
        x = torch.from_numpy(X_train[i]).float().unsqueeze(0).to(device)
        output = model(x)
        _, predicted = torch.max(output, dim=1)
        if predicted.item() == y_train[i]:
            correct += 1

accuracy = correct / len(X_train) * 100
print(f"Training accuracy: {accuracy:.2f}% ({correct}/{len(X_train)})")

Training accuracy: 100.00% (152/152)


In [10]:
data = {
    "model_state": model.state_dict(),
    "input_size": input_size,
    "hidden_size": hidden_size,
    "output_size": output_size,
    "all_words": all_words,
    "tags": tags
}

FILE = "data.pth"
torch.save(data, FILE)
print(f'Training complete. Model saved to {FILE}')

Training complete. Model saved to data.pth


In [11]:
def predict_class(sentence, model, all_words, tags, threshold=0.40):
    tokenized = tokenize(sentence)
    X = bag_of_words(tokenized, all_words)
    X = torch.from_numpy(X).float().unsqueeze(0).to(device)

    output = model(X)
    _, predicted = torch.max(output, dim=1)
    tag = tags[predicted.item()]

    probs = torch.softmax(output, dim=1)
    prob = probs[0][predicted.item()]

    if prob.item() > threshold:
        return tag, prob.item()
    else:
        return "unknown", prob.item()

test_sentences = [
    "Apa jadwal kuliah hari Senin?",
    "Ada mata kuliah apa hari ini?",
    "Mata kuliah dosen Dr. Rina Andayani apa saja?",
    "Kuliah semester 3 apa saja?",
    "Mata kuliah di Laboratorium Komputer Dasar apa saja?",
]

for s in test_sentences:
    tag, prob = predict_class(s, model, all_words, tags)
    print(f"{s!r} -> intent: {tag} (prob={prob:.2f})")

'Apa jadwal kuliah hari Senin?' -> intent: jadwal_hari_tertentu (prob=1.00)
'Ada mata kuliah apa hari ini?' -> intent: jadwal_hari_ini (prob=1.00)
'Mata kuliah dosen Dr. Rina Andayani apa saja?' -> intent: mata_kuliah_dosen (prob=1.00)
'Kuliah semester 3 apa saja?' -> intent: mata_kuliah_per_semester (prob=1.00)
'Mata kuliah di Laboratorium Komputer Dasar apa saja?' -> intent: mata_kuliah_berdasarkan_lokasi (prob=1.00)
